# Multi-factor screening

Apply Benjamini-Hochberg-Yekutieli (BHY) FDR control to a batch of
candidate factors. Demonstrates the explicit-family contract and
the duplicate-identity defense — the behaviour that is impossible to
learn from docstrings alone.

## Factor type

This recipe uses `multi_factor.bhy(...)` over a list of
`EvaluationResult` objects. Each result is produced by
`evaluate(panel, metrics=..., factor_cols=[<name>])` for any registered
cell — screening is factor-type-agnostic.

The input list **is** the family. Each result must carry a unique
identity `(factor, forward_periods)` — the anti-shopping defense. The
recommended path: name the factor column distinctly per candidate panel
and pass it via `factor_cols=[...]`; `evaluate` stamps the `factor` name
onto each returned `EvaluationResult`.

Pass `expand_over=(<context key>,)` to declare per-bucket independent
step-ups (Benjamini & Bogomolov 2014 selective inference). Mixing
`forward_periods` without `expand_over` emits `RuntimeWarning` —
different horizons carry different null distributions and pooling
silently inflates FDR.

Literature: [Benjamini & Yekutieli (2001)](../../reference/bibliography/).

## Use this when

- You have ≥2 candidate factors and want type-I error control under
  multiple testing.
- Candidates are evaluable under the same configuration (same
  scope, density, metric, forward horizon). Mixed cells / horizons
  in one call are caller's responsibility — see § 4 below.
- You prefer FDR control (BHY) over family-wise error (Bonferroni)
  — appropriate when discoveries can tolerate a known false-positive
  rate and power matters.

## What it tests

For an input family of size `N`, BHY step-up keeps results whose
ranked p-values satisfy `p_(k) ≤ q · k / (N · c(N))` where
`c(N) = Σ 1/i` (Benjamini-Yekutieli adjustment for arbitrary
dependence). Pass your nominal `q` directly — the `c(N)` correction
is baked in. Cross-family aggregation is *not* performed; that is
the user's responsibility and is deliberately out of scope.

## Output to read

`bhy` returns a dict of `BhyResult` containers keyed by metric label:

1. `len(survivors)` vs `len(results)` — coarse hit rate.
2. `survivors[i].metrics["ic"].p_value` and `.factor` — which factor
   cleared the family-adjusted bar.
3. `adj_p[i]` — the BHY-adjusted p-value driving the survivor decision
   (survivor membership is definitionally `adj_p <= q`).

## 1. Setup


In [ ]:
from __future__ import annotations

import factrix as fx
import numpy as np
import polars as pl
from factrix.preprocess import compute_forward_return

print("factrix version:", fx.__version__)

## 2. Build a single-family batch

Five candidate factors, all evaluated under Newey-West IC with
`forward_periods=5` — a *valid* BHY input where the step-up actually
controls FDR.

We start from one ground-truth factor and add increasing IID noise
to produce variants with varying signal strengths. Each variant is
materialised under its own column name (`variant_0` … `variant_4`)
so `evaluate(..., factor_cols=[name])` stamps a distinct `factor`
per result — no post-hoc identity surgery needed.

In [ ]:
from factrix.metrics import ic

raw = fx.datasets.make_cs_panel(
    n_assets=100,
    n_dates=500,
    ic_target=0.08,
    seed=2024,
)
panel = compute_forward_return(raw, forward_periods=5)


def variant_panel(
    base: pl.DataFrame, *, name: str, scale: float, seed: int
) -> pl.DataFrame:
    """Add IID noise on top of the ground-truth factor and store under a fresh column name."""
    rng = np.random.default_rng(seed)
    noisy = base["factor"].to_numpy() + scale * rng.standard_normal(base.height)
    return base.drop("factor").with_columns(pl.Series(name, noisy))


candidates = {
    f"variant_{i}": variant_panel(
        panel, name=f"variant_{i}", scale=0.5 + 0.3 * i, seed=100 + i
    )
    for i in range(5)
}
results = []
for name, p in candidates.items():
    results.extend(
        fx.evaluate(
            p,
            metrics={"ic": ic(inference=fx.inference.NEWEY_WEST)},
            factor_cols=[name],
            forward_periods=5,
        ).values()
    )
# Raw p_value only — FDR-controlled decisions wait for BHY in §3.
# Per-factor `p_value < 0.05` thresholding on N candidates is the
# spec-search anti-pattern factrix explicitly avoids.
for res in results:
    print(f"  {res.factor:12s} p_value={res.metrics['ic'].p_value:.4g}")

## 3. Apply BHY

The input list **is** the family. `bhy` runs one Benjamini-Yekutieli
step-up over all results and returns a dict of `BhyResult` containers
keyed by metric label. Each `BhyResult` exposes `.survivors` (in input
order), `.adj_p` (aligned), plus `.q` / `.expand_over` / `.n_tests`
for audit. Survivor membership is definitionally `adj_p <= q`. The
`c(N)` correction is applied internally; pass your nominal `q`.

In [ ]:
bhy_ic = fx.multi_factor.bhy(results, metrics=["ic"], q=0.05)["ic"]
print(f"BHY survivors: {len(bhy_ic.survivors)} / {len(results)}")
for res, adj in zip(bhy_ic.survivors, bhy_ic.adj_p, strict=True):
    print(
        f"  {res.factor:12s} p_value={res.metrics['ic'].p_value:.4g}  adj_p={adj:.4g}"
    )

# BhyResult also renders as a table in Jupyter — see repr output.
bhy_ic

## 4. Duplicate-identity defense

`evaluate()` stamps each result's `factor` from its `factor_cols` name,
so two results built off the same `"factor"` column both land at
identity `("factor", forward_periods)`. Pass two such results to
`bhy()` and the family-resolution layer raises `UserInputError`
rather than silently treating distinct candidates as one hypothesis.

The canonical fix is what § 2 already does: distinct column name
per panel + `evaluate(..., factor_cols=[name])`. Below we deliberately
take the colliding path to surface the error, then show the fix.

In [ ]:
from factrix.metrics import fm_beta, ic

# Colliding path: both calls use factor_cols=["factor"] → identity collide.
unstamped = []
unstamped.extend(
    fx.evaluate(
        panel,
        metrics={"ic": ic(inference=fx.inference.NEWEY_WEST)},
        factor_cols=["factor"],
        forward_periods=5,
    ).values()
)
unstamped.extend(
    fx.evaluate(
        panel, metrics={"fm": fm_beta()}, factor_cols=["factor"], forward_periods=5
    ).values()
)
try:
    fx.multi_factor.bhy(unstamped, metrics=["ic"], q=0.05)
except fx.UserInputError as exc:
    print("UserInputError raised as expected:")
    print(str(exc))

# Canonical fix: rename each panel's factor column upfront so the
# returned results carry distinct `factor` identities.
stamped = []
stamped.extend(
    fx.evaluate(
        panel.rename({"factor": "ic_var"}),
        metrics={"ic": ic(inference=fx.inference.NEWEY_WEST)},
        factor_cols=["ic_var"],
        forward_periods=5,
    ).values()
)
stamped.extend(
    fx.evaluate(
        panel.rename({"factor": "fm_var"}),
        metrics={"fm": fm_beta()},
        factor_cols=["fm_var"],
        forward_periods=5,
    ).values()
)
print(f"\ncanonical fix → identities: {[r.factor for r in stamped]}")

## 5. Where to go next

For the broader `n_assets` × factory behaviour matrix see
[Guides § PANEL vs TIMESERIES](../../guides/panel-timeseries/);
for the BHY contract specifically see
[Guides § Batch screening](../../guides/batch-screening/).


In [ ]:
print("multi_factor_screening: ok")